# Assignment 7: Creating and Fixing Mode Collapse in a GAN

**Dataset:** Fashion-MNIST  |  **Model:** DCGAN (from Assignment 4)

Run in Google Colab with a GPU runtime (Runtime > Change runtime type > T4 GPU), then Runtime > Run all.

## 1. Objective

Mode collapse is a common GAN failure where the generator keeps producing the same few outputs instead of the full variety of the training data. In this assignment I deliberately break the DCGAN from Assignment 4 to cause mode collapse, save the result, apply one fix, retrain, and compare.

## 2. Imports and Setup

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.utils as vutils
import matplotlib.pyplot as plt

torch.manual_seed(42)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

os.makedirs('outputs', exist_ok=True)
print('Device:', device)

## 3. Load the Dataset

Images are scaled to [-1, 1] to match the generator's Tanh output.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
loader = torch.utils.data.DataLoader(dataset, batch_size=128, shuffle=True, drop_last=True)

latent_dim = 100
fixed_noise = torch.randn(64, latent_dim, 1, 1, device=device)

## 4. Model Definitions

### 4.1 Working DCGAN (Assignment 4 style)

In [ ]:
class Generator(nn.Module):
    def __init__(self, base=64, use_bn=True):
        super().__init__()
        def norm(c):
            return nn.BatchNorm2d(c) if use_bn else nn.Identity()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, base * 2, 7, 1, 0, bias=False),
            norm(base * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(base * 2, base, 4, 2, 1, bias=False),
            norm(base),
            nn.ReLU(True),
            nn.ConvTranspose2d(base, 1, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, base=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, base, 4, 2, 1),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(base, base * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base * 2),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(base * 2, 1, 7, 1, 0),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).view(-1)


def init_weights(m):
    name = m.__class__.__name__
    if 'Conv' in name:
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif 'BatchNorm' in name:
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)

## 5. Training and Evaluation Functions

In [ ]:
def train(G, D, lr_g, lr_d, epochs, real_label=1.0):
    criterion = nn.BCELoss()
    opt_g = optim.Adam(G.parameters(), lr=lr_g, betas=(0.5, 0.999))
    opt_d = optim.Adam(D.parameters(), lr=lr_d, betas=(0.5, 0.999))
    g_losses, d_losses = [], []

    for epoch in range(epochs):
        g_sum, d_sum = 0.0, 0.0
        for real, _ in loader:
            real = real.to(device)
            b = real.size(0)
            real_targets = torch.full((b,), real_label, device=device)
            fake_targets = torch.zeros(b, device=device)

            noise = torch.randn(b, latent_dim, 1, 1, device=device)
            fake = G(noise)

            opt_d.zero_grad()
            d_loss = criterion(D(real), real_targets) + criterion(D(fake.detach()), fake_targets)
            d_loss.backward()
            opt_d.step()

            opt_g.zero_grad()
            g_loss = criterion(D(fake), torch.ones(b, device=device))
            g_loss.backward()
            opt_g.step()

            g_sum += g_loss.item()
            d_sum += d_loss.item()

        g_losses.append(g_sum / len(loader))
        d_losses.append(d_sum / len(loader))
        print(f'Epoch {epoch + 1:2d}/{epochs}  D loss: {d_losses[-1]:.4f}  G loss: {g_losses[-1]:.4f}')

    return g_losses, d_losses


def diversity(G):
    G.eval()
    with torch.no_grad():
        imgs = G(fixed_noise).view(64, -1)
        return torch.cdist(imgs, imgs).mean().item()


def show_grid(G, title, path):
    G.eval()
    with torch.no_grad():
        imgs = G(fixed_noise).cpu()
    grid = vutils.make_grid(imgs, nrow=8, padding=2, normalize=True)
    plt.figure(figsize=(7, 7))
    plt.imshow(grid.permute(1, 2, 0))
    plt.title(title)
    plt.axis('off')
    plt.savefig(path, bbox_inches='tight', dpi=150)
    plt.show()


def plot_losses(g_losses, d_losses, title):
    plt.figure(figsize=(7, 3.5))
    plt.plot(g_losses, label='Generator')
    plt.plot(d_losses, label='Discriminator')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.show()

## 6. Baseline Reference

Real Fashion-MNIST images are shown with their diversity score. A healthy generator should score close to this value.

In [ ]:
real_batch = next(iter(loader))[0][:64].to(device)
real_diversity = torch.cdist(real_batch.view(64, -1), real_batch.view(64, -1)).mean().item()
print(f'Diversity of real images: {real_diversity:.2f}')

## 7. Causing Mode Collapse

Modifications made to the working DCGAN:

- Generator capacity reduced (base channels 64 down to 8)
- Batch normalization removed from the generator
- Learning rate raised significantly to 0.01 for both networks

The model is trained for 15 epochs.

In [ ]:
G_bad = Generator(base=8, use_bn=False).to(device)
D_bad = Discriminator().to(device)
G_bad.apply(init_weights)
D_bad.apply(init_weights)

g_bad, d_bad = train(G_bad, D_bad, lr_g=0.01, lr_d=0.01, epochs=15)

In [ ]:
show_grid(G_bad, 'Collapsed generator output (15 epochs)', 'outputs/collapsed_output.png')
plot_losses(g_bad, d_bad, 'Losses with mode collapse setup')
div_bad = diversity(G_bad)
print(f'Diversity score (collapsed): {div_bad:.2f}  |  real images: {real_diversity:.2f}')

## 8. Fixing the Problem

**Technique used: different learning rates for the generator and discriminator.**

The learning rate is lowered from 0.01 to 0.0002 for the generator and 0.0001 for the discriminator. The discriminator learns more slowly so it does not overpower the generator, which gives the generator a stable signal to learn from. The generator and discriminator are retrained from scratch for 15 epochs.

In [ ]:
G_fix = Generator(base=8, use_bn=False).to(device)
D_fix = Discriminator().to(device)
G_fix.apply(init_weights)
D_fix.apply(init_weights)

g_fix, d_fix = train(G_fix, D_fix, lr_g=0.0002, lr_d=0.0001, epochs=15)

In [ ]:
show_grid(G_fix, 'Improved generator output (15 epochs)', 'outputs/improved_output.png')
plot_losses(g_fix, d_fix, 'Losses after applying different learning rates')
div_fix = diversity(G_fix)
print(f'Diversity score (improved): {div_fix:.2f}  |  real images: {real_diversity:.2f}')

## 9. Comparison

In [ ]:
print(f'{"Setting":<28}{"Diversity score":>16}')
print('-' * 44)
print(f'{"Real images (reference)":<28}{real_diversity:>16.2f}')
print(f'{"Collapsed model":<28}{div_bad:>16.2f}')
print(f'{"Improved model":<28}{div_fix:>16.2f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, G, title in zip(axes, [G_bad, G_fix], ['Before: collapsed', 'After: improved']):
    G.eval()
    with torch.no_grad():
        imgs = G(fixed_noise).cpu()
    ax.imshow(vutils.make_grid(imgs, nrow=8, padding=2, normalize=True).permute(1, 2, 0))
    ax.set_title(title)
    ax.axis('off')
plt.savefig('outputs/comparison.png', bbox_inches='tight', dpi=150)
plt.show()

## 10. Written Explanation

*Before submitting, check each answer below against your own output images and diversity scores, and edit the wording if your run looked different.*

### 10.1 What modification caused the model to perform poorly?

I reduced the generator's capacity, removed batch normalization from it, and raised the learning rate to 0.01 for both networks. A small generator without normalization has limited ability to represent many different clothing shapes, and the high learning rate makes the updates large and unstable. The generator finds a few outputs that fool the discriminator and keeps producing them, which is mode collapse.

### 10.2 What did the generated images look like?

The grid showed very little variety. Many images were nearly identical or very similar blurry blobs, and fine details were missing. The diversity score was much lower than the score for real images, and the loss curves were unstable instead of settling into a steady range.

### 10.3 Which technique did you use to improve the model?

I used different learning rates for the generator and the discriminator: 0.0002 for the generator and 0.0001 for the discriminator, both much lower than the 0.01 used in the broken run.

### 10.4 Did the solution improve the output? Why?

Yes, the generated images became more varied and the diversity score moved closer to that of the real images. The smaller step sizes stopped the networks from overshooting each other. A slower discriminator also stops it from winning too quickly, so the generator keeps receiving useful gradients and is pushed to cover more of the data instead of repeating a single output. The images are still not perfect because the generator capacity is small and batch normalization is still removed, which shows that the learning rate setup was the main cause of the instability.